*0.2 Math / ML basics*

# Probability

**The situation.** Support asks why the assistant sometimes answers "Paris" and sometimes "The capital of France is Paris" to the same question. Engineering says "the model is random". That is half the story. The model does not pick words; it produces a *probability for every possible next token* and something else picks from that list.

**Seeing the probabilities.** OpenAI returns them if you ask (`logprobs=True`). They come as *log-probabilities* — the natural log of the probability — because tiny probabilities are easier to handle as logs. `e` to the power of the log gets the probability back.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
import math

from openai import OpenAI

client = OpenAI(timeout=30)
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "The capital of France is"}],
    max_tokens=1,
    logprobs=True,
    top_logprobs=5,  # the five most likely first tokens
)
candidates = response.choices[0].logprobs.content[0].top_logprobs
total = 0.0
for candidate in candidates:
    probability = math.exp(candidate.logprob)
    total += probability
    print(
        
            f"{candidate.token!r:<10} logprob {candidate.logprob:>8.3f}   probability "
            f"{probability:.4%}"
        
    )
print("these five together:", f"{total:.2%}", "— the rest is spread over thousands of other tokens")
assert total <= 1.0001

'The'      logprob   -0.000   probability 99.9565%
'Paris'    logprob   -7.750   probability 0.0431%
'the'      logprob  -12.250   probability 0.0005%
' The'     logprob  -16.000   probability 0.0000%
'par'      logprob  -17.500   probability 0.0000%
these five together: 100.00% — the rest is spread over thousands of other tokens


**Reading the output.** The model's first token is almost certainly one thing (probability near 100%), with a handful of alternatives far behind. The list is the model's actual belief about what comes next. Every sampling setting in the next four items is a rule for choosing from this list.

```
                      ┌ "The"     99.9%
model ──▶ scores ──▶ softmax ──▶ ├ "Paris"    0.07%   ──▶ pick one ──▶ next token
                      ├ "the"     0.003%
                      └ …thousands more
```

**The rule to remember.** A language model outputs a probability distribution over the next token. "Randomness" is a choice made *after* that, by the sampler — and it is yours to set.

| Use it when | Don't when | Instead use |
|---|---|---|
| debugging odd answers, measuring model confidence, building classifiers on top of a model | you only need the answer text | a plain call without logprobs |

**Watch out**
- Logprobs are per token, not per answer. The probability of a whole sentence is the product of its tokens' probabilities — small numbers multiply into tiny ones fast.
- A 99% first token says nothing about whether the *fact* is right; the model can be confidently wrong.
- `top_logprobs` maxes out at 20; you never see the full distribution from a hosted API.